
# Transient simulation — FC-DLC profile

:class:`~marapendi.cell.transient.TransientModel` integrates coupled ODEs for
MEA temperature $T_\mathrm{MEA}(t)$ and the membrane water-content
profile $\lambda(\xi, t)$.

This example drives the cell through one complete **FC-DLC** cycle (JRC EUR
27632 EN, Appendix F) — 35 piecewise-constant steps over 1181 s — under JRC
automotive reference conditions.  The transient dynamics of MEA temperature,
membrane water content, and high-frequency resistance are compared against the
quasi-steady-state (QSS) prediction at every time step.

## Reference
Tsotridis G. et al., "EU Harmonised Test Protocols for PEMFC MEA Testing in
Single Cell Configuration for Automotive Applications", JRC Science for Policy
Report EUR 27632 EN, doi:10.2790/54653 (2015).


## FC-DLC profile and reference conditions



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import marapendi as mrpd
from marapendi.cell.transient import TransientModel
from marapendi.cell.explicit_steady_state import ExplicitSteadyStateModel

# FC-DLC step table: columns [start_time_s, dwell_s, load_%]
_FC_DLC = np.array([
    [0,    15, 0.0  ],
    [15,   13, 12.5 ],
    [28,   33, 5.0  ],
    [61,   35, 26.7 ],
    [96,   47, 5.0  ],
    [143,  20, 41.7 ],
    [163,  25, 29.2 ],
    [188,  22, 5.0  ],
    [210,  13, 12.5 ],
    [223,  33, 5.0  ],
    [256,  35, 26.7 ],
    [291,  47, 5.0  ],
    [338,  20, 41.7 ],
    [358,  25, 29.2 ],
    [383,  22, 5.0  ],
    [405,  13, 12.5 ],
    [418,  33, 5.0  ],
    [451,  35, 26.7 ],
    [486,  47, 5.0  ],
    [533,  20, 41.7 ],
    [553,  25, 29.2 ],
    [578,  22, 5.0  ],
    [600,  13, 12.5 ],
    [613,  33, 5.0  ],
    [646,  35, 26.7 ],
    [681,  47, 5.0  ],
    [728,  20, 41.7 ],
    [748,  25, 29.2 ],
    [773,  68, 5.0  ],
    [841,  58, 58.3 ],
    [899,  82, 41.7 ],
    [981,  85, 58.3 ],
    [1066, 50, 83.3 ],
    [1116, 44, 100.0],
    [1160, 21, 0.0  ],
])
CYCLE_DURATION = 1181  # s

I_MAX      = 10_000.  # A m⁻²  — 100 % load (≈ 0.65 V operating point)
I_MIN_FLOW = 2_000.   # A m⁻²  — minimum gas flow equivalent (JRC §2.2)
I_MIN_SIM  = 100.     # A m⁻²  — floor for OCV steps (avoids division by zero)

# JRC automotive reference conditions (Table 1)
T_CELL  = 353.15   # 80 °C
T_INLET = 358.15   # 85 °C — gas inlet temperature (JRC §2.2 and §2.3)
P_CA    = 2.30e5   # Pa abs — cathode outlet pressure
P_AN    = 2.50e5   # Pa abs — anode outlet pressure
RH_CA   = 0.30    # 30 % RH — cathode inlet
RH_AN   = 0.50    # 50 % RH — anode inlet
ST_CA   = 1.5     # cathode stoichiometry (at I_MIN_FLOW)
ST_AN   = 1.3     # anode stoichiometry (at I_MIN_FLOW)


def fc_dlc_current(t_s):
    """Return FC-DLC current density (A m⁻²) at time *t_s* (seconds)."""
    t_mod = t_s % CYCLE_DURATION
    idx   = np.searchsorted(_FC_DLC[:, 0], t_mod, side='right') - 1
    idx   = int(np.clip(idx, 0, len(_FC_DLC) - 1))
    pct   = _FC_DLC[idx, 2]
    return max(pct / 100.0 * I_MAX, I_MIN_SIM)


def conditions(t):
    """Return CellConditions for the FC-DLC profile at time *t* (seconds)."""
    i = fc_dlc_current(t)
    # Stoichiometry clamped so the gas flow never drops below I_MIN_FLOW equiv.
    st_ca = ST_CA * max(I_MIN_FLOW / i, 1.0)
    st_an = ST_AN * max(I_MIN_FLOW / i, 1.0)
    return mrpd.CellConditions(
        current_density=np.atleast_1d(i),
        cell_temperature=T_CELL,
        ca=mrpd.SideConditions(
            inlet_temperature=T_INLET,
            outlet_pressure=P_CA,
            dry_o2_mole_fraction=0.21,
            inlet_relative_humidity=RH_CA,
            stoichiometry=st_ca,
        ),
        an=mrpd.SideConditions(
            inlet_temperature=T_INLET,
            outlet_pressure=P_AN,
            dry_h2_mole_fraction=1.0,
            inlet_relative_humidity=RH_AN,
            stoichiometry=st_an,
        ),
    )

## Cell assembly



In [ ]:
liq     = mrpd.DarcyTransportModel(J_function_exponent=2)
ionomer = mrpd.PFSAIonomer(equivalent_weight=1100, dry_density=1980)

cell = mrpd.FuelCell(
    area=25e-4,                   # m²
    electrical_resistance=30e-7,  # Ω m²
    ca=mrpd.FuelCellSide(
        cl=mrpd.PtCCatalystLayer(
            ecsa=70e3, platinum_loading=0.4e-2, ionomer=ionomer,
            reaction=mrpd.ElectrochemicalReaction(
                reference_exchange_current_density=2.5e-4,
                reaction_order=0.54, activation_energy=67e6,
                reference_activity=1e5, reference_temperature=353.15,
                number_of_electrons=2, charge_transfer_coeff=0.5,
            ),
            thickness=10e-6, thermal_conductivity=0.22,
            pore_diameter=40e-9, absolute_permeability=1e-13, contact_angle=97.,
            two_phase_transport_model=liq,
        ),
        gdl=mrpd.GasDiffusionLayer(
            thickness=200e-6, porosity=0.6, contact_angle=120.,
            effective_gas_diffusion_ratio=0.3, absolute_permeability=1e-12,
            thermal_conductivity=0.5, two_phase_transport_model=liq,
        ),
        ch=mrpd.FlowChannel(
            width=1e-3, height=1e-3, length=0.1, n_parallel=20, reactant='o2',
        ),
        has_mpl=False, thermal_contact_resistance=4e-4,
    ),
    an=mrpd.FuelCellSide(
        cl=mrpd.PtCCatalystLayer(thickness=5e-6, two_phase_transport_model=liq),
        gdl=mrpd.GasDiffusionLayer(
            thickness=200e-6, effective_gas_diffusion_ratio=0.3,
            thermal_conductivity=0.5, two_phase_transport_model=liq,
        ),
        ch=mrpd.FlowChannel(
            width=1e-3, height=1e-3, length=0.1, n_parallel=20, reactant='h2',
        ),
        has_mpl=False, thermal_contact_resistance=4e-4,
    ),
    membrane=mrpd.PFSA(ionomer=ionomer, dry_thickness=25e-6),
)

## Transient integration

Initialise from the first FC-DLC step (minimum-load conditions) then
integrate for one full cycle.  ``dense_output=True`` allows evaluation
at arbitrary times via ``sol.sol(t)``.



In [ ]:
tr_model = TransientModel(n_memb_mesh=3)
_, x0    = tr_model.set_initial_conditions(cell, conditions(0))

sol = tr_model.solve(cell, conditions, t_span=(0, CYCLE_DURATION),
                     x0=x0, dense_output=True)
print(f"ODE status: {sol.status}  ({sol.message})")
print(f"Number of ODE steps: {len(sol.t)}")

## Evaluate diagnostics on a regular time grid

:meth:`~marapendi.cell.transient.TransientModel.evaluate` accepts the
``conditions`` callable directly — it is called at every evaluation time step.



In [ ]:
t_eval = np.linspace(0, CYCLE_DURATION, 400)
diag   = tr_model.evaluate(cell, conditions, t_eval, x_eval=sol.sol(t_eval))

## Quasi-steady reference

Run the same conditions through the quasi-steady model to compare with the
fully-resolved transient: any discrepancy reveals how much the MEA thermal
and water-transport inertia matter over the FC-DLC timescale.



In [ ]:
i_arr  = np.array([fc_dlc_current(t) for t in t_eval])
st_ca_arr = ST_CA * np.maximum(I_MIN_FLOW / i_arr, 1.0)
st_an_arr = ST_AN * np.maximum(I_MIN_FLOW / i_arr, 1.0)

qss_cond = mrpd.CellConditions(
    current_density=i_arr,
    cell_temperature=T_CELL,
    ca=mrpd.SideConditions(
        inlet_temperature=T_INLET, outlet_pressure=P_CA,
        dry_o2_mole_fraction=0.21, inlet_relative_humidity=RH_CA,
        stoichiometry=st_ca_arr,
    ),
    an=mrpd.SideConditions(
        inlet_temperature=T_INLET, outlet_pressure=P_AN,
        dry_h2_mole_fraction=1.0, inlet_relative_humidity=RH_AN,
        stoichiometry=st_an_arr,
    ),
)
ss_model = ExplicitSteadyStateModel()
ss_state = ss_model.solve(cell, qss_cond,
                          ss_model.set_initial_conditions(cell, qss_cond))
hfr_qss  = ss_model.voltage_model.high_frequency_resistance(cell, ss_state)

## FC-DLC current profile



In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.fill_between(t_eval, i_arr * 1e-4, step='post', alpha=0.30, color='C0')
ax.step(t_eval, i_arr * 1e-4, where='post', color='C0', lw=1.2)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Current density (A cm⁻²)")
ax.set_title("JRC FC-DLC — one cycle (EUR 27632 EN, App. F)")
ax.set_xlim(0, CYCLE_DURATION)
ax.grid(True, alpha=0.3)
fig.tight_layout()

## Transient vs quasi-steady response



In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)

# -- MEA temperature --
ax = axes[0, 0]
ax.plot(t_eval, diag.mea_temperature - 273.15, 'C0', lw=1.5, label="Transient")
ax.plot(t_eval, ss_state.mea_temperature - 273.15, 'C0--', lw=1.0, alpha=0.7,
        label="QSS")
ax.set_ylabel("T_MEA (°C)")
ax.set_title("MEA temperature")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# -- Membrane water content --
ax = axes[0, 1]
ax.plot(t_eval, diag.membrane.water_content, 'C1', lw=1.5, label="Transient")
ax.plot(t_eval, ss_state.membrane.water_content, 'C1--', lw=1.0, alpha=0.7,
        label="QSS")
ax.set_ylabel("λ_membrane (mol/mol)")
ax.set_title("Mean membrane water content")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# -- HFR --
ax = axes[1, 0]
ax.plot(t_eval, np.asarray(diag.hfr) * 1e4, 'C2', lw=1.5, label="Transient")
ax.plot(t_eval, hfr_qss * 1e4, 'C2--', lw=1.0, alpha=0.7, label="QSS")
ax.set_xlabel("Time (s)")
ax.set_ylabel("HFR (mΩ cm²)")
ax.set_title("High-frequency resistance")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# -- Cell voltage --
ax = axes[1, 1]
ax.plot(t_eval, diag.cell_voltage, 'C3', lw=1.5, label="Transient")
ax.plot(t_eval, ss_state.cell_voltage, 'C3--', lw=1.0, alpha=0.7, label="QSS")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Cell voltage (V)")
ax.set_title("Cell voltage")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

fig.suptitle("Transient vs quasi-steady response — JRC FC-DLC", fontsize=11)
fig.tight_layout()

## Membrane water-content profile snapshots

The through-plane water-content profile λ(ξ, t) evolves as water is
redistributed by electroosmotic drag and back-diffusion between FC-DLC steps.



In [ ]:
xi     = np.linspace(0, 1, tr_model.n_memb_mesh)
# Pick one representative time per FC-DLC step (mid-point of each dwell)
t_snap = _FC_DLC[:, 0] + _FC_DLC[:, 1] / 2.0
load_snap = _FC_DLC[:, 2]

fig, ax = plt.subplots(figsize=(7, 4.5))
cmap = plt.cm.plasma
norm = plt.Normalize(vmin=0, vmax=100)
sm   = plt.cm.ScalarMappable(cmap=cmap, norm=norm)

for t_s, load_pct in zip(t_snap, load_snap):
    profile = sol.sol(t_s)[1:]   # λ at each mesh node
    ax.plot(xi, profile, '-', lw=1.0, color=cmap(norm(load_pct)), alpha=0.8)

plt.colorbar(sm, ax=ax, label="FC-DLC load (%)")
ax.set_xlabel("Dimensionless position ξ  (0 = anode, 1 = cathode)")
ax.set_ylabel("Water content λ (mol/mol)")
ax.set_title("Membrane water-content profiles at mid-point of each FC-DLC step")
ax.grid(True, alpha=0.3)
fig.tight_layout()